In [ ]:
from dotenv import load_dotenv

load_dotenv(
)

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5-nano")
# model.invoke("hello")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory, ConfigurableFieldSpec
from langchain_core.messages import SystemMessage
from InMemoryChatMessageHistory import get_history
from langchain_core.chat_history import BaseChatMessageHistory

store = {}

def get_session_history(user_id: str, session_id: str) -> BaseChatMessageHistory:
    user_session_tuple = (user_id, session_id)
    if user_session_tuple not in store:
        store[user_session_tuple] = get_history()
    return store[user_session_tuple]

prompt = ChatPromptTemplate.from_messages(
            [
                SystemMessage("You're an assistant who's good at Mathematics." +
                                  "Give explanation only when asked otherwise ouput the solution"),
                MessagesPlaceholder(variable_name="history"),
                ("human", "{question}"),
            ]
        )

chain = prompt | model

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="user_id",
            annotation=str,
            name="User ID",
            description="User ID of the current user",
        ),
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="User ID",
            description="Session id of the current chat session",
        )
    ]
)

config={"configurable": {"session_id": "1", "user_id": "1"}}

chain_with_history.invoke({"question": "What is 2+1?"}, config=config)